In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

process = Path("../Data/Processed")

demand_file = process / "retail_product_daily_demand.csv"
churn_file = process / "retail_churn_features.csv"

demand_df = pd.read_csv(demand_file,parse_dates=["Date"])
churn_df = pd.read_csv(churn_file)

print(demand_df.columns.tolist())
print(churn_df.columns.tolist())

['StockCode', 'Date', 'units_sold', 'revenue', 'n_invoices', 'units_roll_7', 'units_roll_14', 'units_roll_30', 'rev_roll_7', 'rev_roll_14', 'rev_roll_30', 'units_lag_1', 'units_lag_7', 'units_lag_14', 'day_of_week', 'day_of_month', 'month', 'quarter', 'is_weekend', 'is_dec']
['CustomerID', 'PrimaryCountry', 'Frequency', 'Monetary', 'TotalQuantity', 'AverageOrderValue', 'CustomerLifetimeDays', 'FrequencyScore', 'MonetaryScore', 'purchase_rate', 'revenue_per_day', 'qty_per_order', 'avg_gap', 'spend_per_visit', 'is_one_time', 'qty_per_day', 'is_slow_buyer', 'aov_ratio', 'Churned']


In [7]:
from scipy.stats import ks_2samp

monitor_cols = ["units_sold","revenue","n_invoices","units_roll_7","units_roll_14","units_roll_30",
    "units_lag_1","units_lag_7","units_lag_14"]

complete_df = demand_df[demand_df["Date"] < demand_df["Date"].max()].copy()

last_date = complete_df["Date"].max()

current_start = last_date-pd.Timedelta(days=59)
reference_start = current_start-pd.Timedelta(days=60)

reference_df = complete_df[(complete_df["Date"] >= reference_start)&(complete_df["Date"] < current_start)]

current_df = complete_df[complete_df["Date"] >= current_start]

drift_results = []

for col in monitor_cols:
    _,p_value = ks_2samp(reference_df[col],current_df[col])

    drift_results.append({
        "Feature":col,
        "P_Value":round(p_value,4),
        "Drifted":p_value < 0.05
    })

drift_df = pd.DataFrame(drift_results)

drift_score = drift_df["Drifted"].mean()

print(drift_df)
print("\nDrift score:",round(drift_score,2))
print("Drifted columns:",drift_df["Drifted"].sum(),"/",len(monitor_cols))

         Feature  P_Value  Drifted
0     units_sold   0.0429     True
1        revenue   0.0794    False
2     n_invoices   0.0027     True
3   units_roll_7   0.1802    False
4  units_roll_14   0.1214    False
5  units_roll_30   0.0089     True
6    units_lag_1   0.1057    False
7    units_lag_7   0.1214    False
8   units_lag_14   0.4416    False

Drift score: 0.33
Drifted columns: 3 / 9


our score is less than 0.5 so we don't need to retrain our model

In [15]:
from pathlib import Path

log_dir = Path("../Reports/Drift_Reports")
drift_df.to_csv(log_dir / "13-Drift_Summary.csv",index=False)

In [18]:
from datetime import datetime

log_row = pd.DataFrame([{
    "RunDate":datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "DriftScore":round(drift_score,2),
    "DriftedColumns":int(drift_df["Drifted"].sum()),
    "TotalColumns":len(monitor_cols),
    "RetrainRequired":drift_score >= 0.50,
    "Decision":"Retrain model" if drift_score >= 0.50 else "Skip retraining"
}])

log_file = log_dir / "13-Retraining_Log.csv"

if log_file.exists():
    old_log = pd.read_csv(log_file)
    log_row = pd.concat([old_log,log_row],ignore_index=True)

log_row.to_csv(log_file,index=False)

print("Retraining log saved.")
log_row.tail()

Retraining log saved.


,RunDate,DriftScore,DriftedColumns,TotalColumns,RetrainRequired,Decision
0,2026-06-12 21:49:30,0.33,3,9,False,Skip retraining
